# 0. Libraries 

In [ ]:
# ==============================
# Core libraries
# ==============================
import numpy as np
import pandas as pd
import time
import warnings
import multiprocessing
from datetime import date, datetime, timedelta

# Use all but one CPU core for parallel processing
num_cores = max(multiprocessing.cpu_count() - 1, 1)
print("Using", num_cores, "cores for parallel processing.")

warnings.filterwarnings("ignore")


# ==============================
# Visualisation
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")  # optional, just to make plots look nicer


# ==============================
# Statistical utilities (EDA, tests)
# ==============================
import scipy.stats as stats
from scipy.stats import chi2, chi2_contingency, f_oneway

# Statsmodels (OLS, ANOVA, etc.)
import statsmodels.api as sm
from statsmodels.formula.api import ols


# ==============================
# Preprocessing & feature engineering
# ==============================
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Optional: dimensionality reduction
from sklearn.decomposition import PCA


# ==============================
# Modelling algorithms (base models)
# ==============================

# Linear models / GLM-style
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

# Tree-based and ensemble models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    StackingRegressor,
    VotingRegressor
)

from sklearn.tree import DecisionTreeRegressor

# Distance-based & kernel-based models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Neural network regressor
from sklearn.neural_network import MLPRegressor


# ==============================
# Gradient boosting libraries (external)
# ==============================
# XGBoost
try:
    import xgboost as xgb
    xgb_available = True
    print("XGBoost available.")
except ImportError:
    xgb_available = False
    print("XGBoost NOT available (install xgboost if you want to use it).")

# LightGBM
try:
    import lightgbm as lgb
    lgb_available = True
    print("LightGBM available.")
except ImportError:
    lgb_available = False
    print("LightGBM NOT available (install lightgbm if you want to use it).")

# CatBoost (optional; often slower but nice to try)
try:
    import catboost as cb
    cb_available = True
    print("CatBoost available.")
except ImportError:
    cb_available = False
    print("CatBoost NOT available (install catboost if you want to use it).")


# ==============================
# Model evaluation & selection
# ==============================
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ==============================
# Model interpretation tools
# ==============================
from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay
)

# If you later want SHAP, you can add:
# import shap


# 1. Loading the Data & Basic Inspection

In [ ]:
# Importing the training dataset
train = pd.read_csv("ML_WP_data/train.csv")

# Basic structural info (includes shape, dtypes, non-null counts)
train.info()

# Quick look at the first rows
display(train.head())

# Calculate total number of NaN values in the DataFrame
total_train_nans = train.isna().sum().sum()
print("Total NaN values in the training DataFrame:", total_train_nans)

# ------------------------------
# Helper: missingness summary (train only)
# ------------------------------

def missing_summary(df, sort_by="Percent_Missing", ascending=False):
    """
    Build a table with:
    - Data type
    - Number of missing values
    - Percentage of missing values
    - Basic descriptive stats (mean, std, min, 25%, 50%, 75%, max) for numeric cols
    """

    n_rows = len(df)

    # Core missing-value info
    miss = df.isna().sum()
    miss = miss[miss > 0]  # keep only variables with at least one missing

    summary = pd.DataFrame({
        "Data_Type": df[miss.index].dtypes.astype(str),
        "Missing_Values": miss,
        "Percent_Missing": (miss / n_rows * 100).round(2)
    })

    # Descriptive stats for numeric columns
    numeric_cols = df[miss.index].select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        desc = df[numeric_cols].describe().T[
            ["mean", "std", "min", "25%", "50%", "75%", "max"]
        ]
        summary = summary.join(desc, how="left")

    # Sort and print total
    summary = summary.sort_values(sort_by, ascending=ascending)
    print(f"Total variables with missing values: {summary.shape[0]}")

    return summary

# Compute missingness summary for train
missing_train = missing_summary(train)
display(missing_train.head(70))  # show first 70 rows; adjust as needed


# 2. Target columns definition

In [ ]:
target_cols = [
    "target_tre200h0_plus12h",
    "target_tre200h0_plus24h",
    "target_tre200h0_plus48h"
]
print("Target columns:", target_cols)
print("Missing values per target:")
print(train[target_cols].isna().sum(), "\n")


# 3. Exploratory Data Analysis (EDA)

## 3.1 Missingness & descriptive statistics

In [ ]:
# Summary for variables with missing values
missing_summary = (
    pd.DataFrame({
        "Data_Type": train.dtypes,
        "Missing_Values": train.isnull().sum(),
        "Percent_Missing": (train.isnull().sum() / len(train) * 100).round(2)
    })
    .query("Missing_Values > 0")
    .sort_values(by="Missing_Values", ascending=False)
)

# Descriptive stats
summary_stats = train.describe(include="all").transpose()

# Merge missingness with basic stats
merged_summary = missing_summary.merge(
    summary_stats[["mean", "std", "min", "25%", "50%", "75%", "max"]],
    left_index=True,
    right_index=True,
    how="left"
)

print(merged_summary.to_string())
print(f"\nTotal variables with missing values: {len(missing_summary)}")


## 3.2 Outlier overview (numeric variables, excluding targets)

In [ ]:
# Numeric columns (excluding targets)
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in target_cols]

def detect_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    return outliers

outlier_summary = {}
for col in numeric_cols:
    n_outliers = len(detect_outliers(train, col))
    outlier_summary[col] = n_outliers

outlier_df = pd.DataFrame(list(outlier_summary.items()), columns=["Variable", "Outlier_Count"])
outlier_df["Outlier_%"] = (outlier_df["Outlier_Count"] / len(train)) * 100
outlier_df.sort_values(by="Outlier_%", ascending=False, inplace=True)

display(outlier_df.head(20))


## 3.3 Target distributions (histogram, boxplot, Q-Q plot)

In [ ]:
fig, axes = plt.subplots(len(target_cols), 3, figsize=(15, 12))

for i, target_col in enumerate(target_cols):
    # Histogram
    axes[i, 0].hist(train[target_col].dropna(), bins=50, edgecolor="black", alpha=0.7)
    axes[i, 0].set_title(f"{target_col} – Distribution")
    axes[i, 0].set_xlabel("Temperature (°C)")
    axes[i, 0].set_ylabel("Frequency")

    # Boxplot
    axes[i, 1].boxplot(train[target_col].dropna(), vert=True)
    axes[i, 1].set_title(f"{target_col} – Boxplot")
    axes[i, 1].set_ylabel("Temperature (°C)")

    # Q–Q plot vs normal
    stats.probplot(train[target_col].dropna(), dist="norm", plot=axes[i, 2])
    axes[i, 2].set_title(f"{target_col} – Q–Q Plot vs Normal")

plt.tight_layout()
plt.show()


## 3.4 Skewness and kurtosis of targets

In [ ]:
for t in target_cols:
    clean_series = train[t].dropna()
    clean_series = clean_series[np.isfinite(clean_series)]

    skew = stats.skew(clean_series)
    kurt = stats.kurtosis(clean_series)

    print(f"{t}: Skewness = {skew:.3f}, Kurtosis = {kurt:.3f}")


## 3.5 Missingness by hour (pattern + chi-square test)

In [ ]:
# Row-wise missing count
train["missing_count"] = train.isnull().sum(axis=1)

# Total missing values per hour
missing_by_hour = (
    train.groupby("hour")["missing_count"]
    .sum()
    .sort_values(ascending=False)
)

print("Top hours with most missing values:\n")
print(missing_by_hour.head(10))

# Plot
plt.figure(figsize=(10, 4))
missing_by_hour.sort_index().plot(kind="bar", edgecolor="black")
plt.title("Total Missing Values by Hour of the Day")
plt.xlabel("Hour (0–23)")
plt.ylabel("Number of Missing Values")
plt.tight_layout()
plt.show()

# Chi-square goodness-of-fit: are missing values equally distributed by hour?
missing_by_hour = train.groupby("hour")["missing_count"].sum()
expected = [missing_by_hour.sum() / len(missing_by_hour)] * len(missing_by_hour)

chi2_stat = ((missing_by_hour - expected) ** 2 / expected).sum()
p_value = 1 - chi2.cdf(chi2_stat, df=len(missing_by_hour) - 1)

print(f"Chi-square statistic: {chi2_stat:.2f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("→ Missingness differs significantly by hour (reject H₀).")
else:
    print("→ No significant hourly difference (fail to reject H₀).")


## 3.6 Missingness by season (if available) + chi-square + ANOVA

In [ ]:

import matplotlib.pyplot as plt
from scipy.stats import chi2

# ------------------------------------------------------------
# Row‑wise missing count
# ------------------------------------------------------------
train["missing_count"] = train.isnull().sum(axis=1)

# ------------------------------------------------------------
# Total missing values per hour (absolute)
# ------------------------------------------------------------
missing_by_hour = (
    train.groupby("hour")["missing_count"]
    .sum()
    .sort_values(ascending=False)          # highest‑missing hours first
)

# ------------------------------------------------------------
# Relative quantity (percentage of all missing values)
# ------------------------------------------------------------
total_missing = missing_by_hour.sum()
missing_pct = (missing_by_hour / total_missing * 100).round(2)

# Build a small table that shows both absolute and relative numbers
missing_summary = pd.DataFrame({
    "missing_abs": missing_by_hour,
    "missing_pct": missing_pct
})

print("\nTop hours with most missing values (absolute & % of total):\n")
print(missing_summary.head(10))

# ------------------------------------------------------------
# Plot absolute counts
# ------------------------------------------------------------
plt.figure(figsize=(10, 4))
missing_by_hour.sort_index().plot(kind="bar", edgecolor="black")
plt.title("Total Missing Values by Hour of the Day")
plt.xlabel("Hour (0–23)")
plt.ylabel("Number of Missing Values")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Chi‑square goodness‑of‑fit: are missing values equally distributed by hour?
# ------------------------------------------------------------
expected = [total_missing / len(missing_by_hour)] * len(missing_by_hour)

chi2_stat = ((missing_by_hour - expected) ** 2 / expected).sum()
p_value = 1 - chi2.cdf(chi2_stat, df=len(missing_by_hour) - 1)

print(f"\nChi‑square statistic: {chi2_stat:.2f}")
print(f"p‑value: {p_value:.4f}")

if p_value < 0.05:
    print("→ Missingness differs significantly by hour (reject H₀).")
else:
    print("→ No significant hourly difference (fail to reject H₀).")


## 3.7 Distributions of variables with most outliers

In [ ]:
# Take the top k variables with highest outlier percentage
k = 13  # adjust as you like
top_outlier_vars = outlier_df.head(k)["Variable"].tolist()

print("Variables with highest outlier %:")
print(top_outlier_vars)

for col in top_outlier_vars:
    series = train[col].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Outlier inspection – {col}", fontsize=12)

    # Histogram
    axes[0].hist(series, bins=50, edgecolor="black", alpha=0.7)
    axes[0].set_title("Histogram")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")

    # Boxplot
    axes[1].boxplot(series, vert=True)
    axes[1].set_title("Boxplot")
    axes[1].set_ylabel(col)

    plt.tight_layout()
    plt.show()


### log10 histograms for non-negative, highly skewed variables among top outliers

In [ ]:
log_candidates = []
for col in top_outlier_vars:
    s = train[col].dropna()
    if (s >= 0).all() and s.max() > 0:
        log_candidates.append(col)

print("Log-scale candidates (non-negative among top outliers):")
print(log_candidates)

for col in log_candidates:
    s = train[col].dropna()
    s_pos = s[s > 0]  # avoid log(0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Raw vs log10 – {col}", fontsize=12)

    axes[0].hist(s, bins=50, edgecolor="black", alpha=0.7)
    axes[0].set_title("Raw scale")
    axes[0].set_xlabel(col)

    axes[1].hist(np.log10(s_pos), bins=50, edgecolor="black", alpha=0.7)
    axes[1].set_title("log10 scale (values > 0)")
    axes[1].set_xlabel(f"log10({col})")

    plt.tight_layout()
    plt.show()


# 4. Feature engineering: advanced weather features

In [ ]:
def create_advanced_features(df, is_training=True):
    """
    Create advanced features for weather prediction.

    Parameters
    ----------
    df : DataFrame
        Input dataframe
    is_training : bool
        Whether this is training data (affects which features we can create)

    Returns
    -------
    DataFrame with new features
    """
    df = df.copy()

    # ============================================================
    # 1. CYCLICAL ENCODING (time-of-day, season)
    # ============================================================
    if "hour" in df.columns:
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
        df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)
        df["is_morning"] = ((df["hour"] >= 6) & (df["hour"] <= 12)).astype(int)
        df["is_afternoon"] = ((df["hour"] >= 12) & (df["hour"] <= 18)).astype(int)
        df.drop(columns=["hour"], inplace=True)

    if "season" in df.columns:
        season_order = ["winter", "spring", "summer", "autumn"]
        season_to_num = {s: i for i, s in enumerate(season_order)}

        df["season_num"] = df["season"].astype(str).str.lower().map(season_to_num)
        df["season_sin"] = np.sin(2 * np.pi * df["season_num"] / 4)
        df["season_cos"] = np.cos(2 * np.pi * df["season_num"] / 4)
        df["is_winter"] = (df["season_num"] == 0).astype(int)
        df["is_summer"] = (df["season_num"] == 2).astype(int)
        df.drop(columns=["season", "season_num"], inplace=True)

    # ============================================================
    # 2. STATION AGGREGATIONS
    # ============================================================
    # Temperature features (tre200h0)
    temp_cols = [c for c in df.columns if "tre200h0_" in c and c != "tre200h0" and "lag" not in c]
    if len(temp_cols) > 0:
        df["temp_mean_all_stations"] = df[temp_cols].mean(axis=1)
        df["temp_std_all_stations"] = df[temp_cols].std(axis=1)
        df["temp_min_all_stations"] = df[temp_cols].min(axis=1)
        df["temp_max_all_stations"] = df[temp_cols].max(axis=1)
        df["temp_range_all_stations"] = df["temp_max_all_stations"] - df["temp_min_all_stations"]

        if "tre200h0" in df.columns:
            df["temp_current_vs_mean"] = df["tre200h0"] - df["temp_mean_all_stations"]

        if "tre200h0_lag24h" in df.columns and "tre200h0" in df.columns:
            df["temp_change_24h"] = df["tre200h0"] - df["tre200h0_lag24h"]
            df["temp_change_24h_abs"] = np.abs(df["temp_change_24h"])

    # Humidity features (ure200h0)
    humidity_cols = [c for c in df.columns if "ure200h0_" in c]
    if len(humidity_cols) > 0:
        df["humidity_mean"] = df[humidity_cols].mean(axis=1)
        df["humidity_std"] = df[humidity_cols].std(axis=1)
        df["humidity_max"] = df[humidity_cols].max(axis=1)
        df["humidity_min"] = df[humidity_cols].min(axis=1)

    # Pressure features (prestah0)
    pressure_cols = [c for c in df.columns if "prestah0_" in c]
    if len(pressure_cols) > 0:
        df["pressure_mean"] = df[pressure_cols].mean(axis=1)
        df["pressure_std"] = df[pressure_cols].std(axis=1)
        df["pressure_range"] = df[pressure_cols].max(axis=1) - df[pressure_cols].min(axis=1)

    # Wind features (fkl010h0, fkl010h3)
    wind_current_cols = [c for c in df.columns if "fkl010h0_" in c]
    wind_3h_cols = [c for c in df.columns if "fkl010h3_" in c]
    if len(wind_current_cols) > 0:
        df["wind_mean_current"] = df[wind_current_cols].mean(axis=1)
        df["wind_max_current"] = df[wind_current_cols].max(axis=1)
    if len(wind_3h_cols) > 0:
        df["wind_mean_3h"] = df[wind_3h_cols].mean(axis=1)
        df["wind_max_3h"] = df[wind_3h_cols].max(axis=1)
    if len(wind_current_cols) > 0 and len(wind_3h_cols) > 0:
        df["wind_change_3h"] = df["wind_mean_3h"] - df["wind_mean_current"]

    # Precipitation features (rre150h0)
    precip_cols = [c for c in df.columns if "rre150h0_" in c]
    if len(precip_cols) > 0:
        df["precip_total"] = df[precip_cols].sum(axis=1)
        df["precip_max"] = df[precip_cols].max(axis=1)
        df["precip_stations_active"] = (df[precip_cols] > 0).sum(axis=1)

    # Radiation features (gre000h0)
    radiation_cols = [c for c in df.columns if "gre000h0_" in c]
    if len(radiation_cols) > 0:
        df["radiation_mean"] = df[radiation_cols].mean(axis=1)
        df["radiation_max"] = df[radiation_cols].max(axis=1)
        df["radiation_std"] = df[radiation_cols].std(axis=1)

    # ============================================================
    # 3. INTERACTION FEATURES
    # ============================================================
    if "temp_mean_all_stations" in df.columns and "humidity_mean" in df.columns:
        df["temp_humidity_interaction"] = df["temp_mean_all_stations"] * df["humidity_mean"]

    if "pressure_mean" in df.columns and "wind_mean_current" in df.columns:
        df["pressure_wind_interaction"] = df["pressure_mean"] * df["wind_mean_current"]

    if "temp_mean_all_stations" in df.columns and "radiation_mean" in df.columns:
        df["temp_radiation_interaction"] = df["temp_mean_all_stations"] * df["radiation_mean"]

    # ============================================================
    # 4. MISSING DATA FEATURES
    # ============================================================
    df["missing_count"] = df.isna().sum(axis=1)
    df["missing_temp_stations"] = df[temp_cols].isna().sum(axis=1) if len(temp_cols) > 0 else 0

    return df


# 5. Building dataset variants for modelling

## 5.1 Outliser removal

HERE OUTLIER REMOVAL¡¡¡

## 5.2 Creation of the datasets

In [ ]:
# 1) Apply advanced feature engineering to the raw train data
train_fe = create_advanced_features(train, is_training=True)

# If you have an outlier-removal step, apply it HERE on train_fe
# e.g. train_clean = remove_outliers(train_fe)
# For now, just copy:
train_clean = train_fe.copy()

print("Original train shape:", train.shape)
print("After feature engineering (train_fe):", train_fe.shape)
print("After optional cleaning (train_clean):", train_clean.shape)

# --- Step 1: Preserve original and define base dataset ---
base = train_clean.copy()

# Identify numeric and categorical columns in the dataset
num_cols = base.select_dtypes(include="number").columns
cat_cols = base.select_dtypes(exclude="number").columns

print(f"Numeric columns: {len(num_cols)}, Categorical columns: {len(cat_cols)}")

# 1) Drop rows with any NA values  → main 'drop_na' dataset
train_drop = base.dropna().copy()

# 2) Mean-imputed dataset
mean_imputer = SimpleImputer(strategy="mean")
train_mean_imp = base.copy()
train_mean_imp[num_cols] = mean_imputer.fit_transform(train_mean_imp[num_cols])

if len(cat_cols) > 0:
    cat_modes_mean = train_mean_imp[cat_cols].mode().iloc[0]
    train_mean_imp[cat_cols] = train_mean_imp[cat_cols].fillna(cat_modes_mean)

# 3) Median-imputed dataset
median_imputer = SimpleImputer(strategy="median")
train_median_imp = base.copy()
train_median_imp[num_cols] = median_imputer.fit_transform(train_median_imp[num_cols])

if len(cat_cols) > 0:
    cat_modes_median = train_median_imp[cat_cols].mode().iloc[0]
    train_median_imp[cat_cols] = train_median_imp[cat_cols].fillna(cat_modes_median)

# 4) KNN imputation (more sophisticated)
print("\nPerforming KNN imputation (this may take a moment)...")
train_knn_imp = base.copy()

knn_numeric_cols = train_knn_imp.select_dtypes(include=[np.number]).columns
knn_numeric_data = train_knn_imp[knn_numeric_cols]

knn_imputer = KNNImputer(n_neighbors=5, weights="distance")
train_knn_imp[knn_numeric_cols] = knn_imputer.fit_transform(knn_numeric_data)

if len(cat_cols) > 0:
    cat_modes_knn = train_knn_imp[cat_cols].mode().iloc[0]
    train_knn_imp[cat_cols] = train_knn_imp[cat_cols].fillna(cat_modes_knn)

print(f"\nknn_imputed shape: {train_knn_imp.shape}")
print("\nAll dataset variants created successfully!")

# Sanity checks
print(f"\nRows before drop: {len(base)}, after drop: {len(train_drop)}")
print("Remaining NAs after drop:", train_drop.isnull().sum().sum())
print("Remaining NAs after mean imputation:", train_mean_imp.isnull().sum().sum())
print("Remaining NAs after median imputation:", train_median_imp.isnull().sum().sum())
print("Remaining NAs after KNN imputation:", train_knn_imp.isnull().sum().sum())

# --- Step 2: Build datasets dictionary (NO scaling, NO PCA) ---
datasets = {
    "drop_na":        train_drop,
    "mean_imputed":   train_mean_imp,
    "median_imputed": train_median_imp,
    "knn_imputed":    train_knn_imp,
}

# Quick sanity check on cyclical features in the main dataset
if {"hour_sin", "hour_cos"}.issubset(datasets["drop_na"].columns):
    display(datasets["drop_na"][["hour_sin", "hour_cos"]].head())


# 6. Creation of Baseline models (24h horizon)

In [ ]:
# Map horizons to target columns
horizons = {
    "12h": "target_tre200h0_plus12h",
    "24h": "target_tre200h0_plus24h",
    "48h": "target_tre200h0_plus48h"
}

# Our main challenge horizon
current_horizon = "24h"
target_col = horizons[current_horizon]
print(f"Current horizon: {current_horizon}, target column: {target_col}")

# Use the feature-engineered, drop_na dataset
df_model = datasets["drop_na"].copy()

X = df_model.drop(columns=list(horizons.values()))  # all features
y = df_model[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
print("Train size:", X_train.shape[0], "Validation size:", X_valid.shape[0])


## 6.1 Preprocessor and model zoo (default hyperparameters)

In [ ]:
# Numeric features (everything should be numeric after feature engineering)
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ],
    remainder="drop"
)

# Define baseline models (default or near-default)
models = {}

models["LinearRegression"] = LinearRegression()
models["Ridge"] = Ridge()
models["Lasso"] = Lasso(max_iter=5000)
models["RandomForest"] = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    criterion="absolute_error"
)
models["ExtraTrees"] = ExtraTreesRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
models["GradientBoosting"] = GradientBoostingRegressor(random_state=42)
models["KNN"] = KNeighborsRegressor()
models["SVR_rbf"] = SVR(kernel="rbf")
models["MLP"] = MLPRegressor(random_state=42, max_iter=300)

if "xgb_available" in globals() and xgb_available:
    models["XGBRegressor"] = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        random_state=42,
        n_jobs=num_cores
    )

if "lgb_available" in globals() and lgb_available:
    models["LGBMRegressor"] = lgb.LGBMRegressor(
        n_estimators=500,
        objective="regression",
        random_state=42
    )

# Optional: CatBoost if installed (all features numeric, so basic use is fine)
if "cb_available" in globals() and cb_available:
    models["CatBoostRegressor"] = cb.CatBoostRegressor(
        verbose=0,
        random_state=42,
        loss_function="MAE"
    )


## 6.2 Run Stage 1: fit all models with default parameters

In [ ]:

results = []

for name, reg in models.items():
    print(f"\nTraining {name} ...")
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", reg)
    ])

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_valid)
    mae = mean_absolute_error(y_valid, y_pred)
    mse = mean_squared_error(y_valid, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_valid, y_pred)


    results.append({
        "model": name,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "train_time_sec": train_time
    })

# Convert results to DataFrame and sort by MAE
results_df = pd.DataFrame(results).sort_values(by="mae").reset_index(drop=True)

# Rounding for a nicer display
results_df = results_df.round(3)

# Display results
display(results_df)

# Get Top 5 models by MAE
top6 = results_df.head(6)["model"].tolist()
print("\nTop 6 models by MAE for 24h:", top6)


### 6.2.2 Testing agains a dumb model

In [ ]:
df = datasets["drop_na"].copy()
target_col = "target_tre200h0_plus24h"

baseline_pred = df["tre200h0_lag24h"]  # or "tre200h0"
mae_baseline = mean_absolute_error(df[target_col], baseline_pred)
print("Baseline MAE (lag24h):", mae_baseline)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET_COL = "tre200h0"  # change if your target has a different name

# 1) Create features for the whole dataset
df_feat = create_advanced_features(df_raw, is_training=True)

# 2) Split into X (features) and y (target)
X = df_feat.drop(columns=[TARGET_COL])
y = df_feat[TARGET_COL]

# 3) Random train–test split (since you only have hour/season, no real date)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

# 4) Build Ridge pipeline
ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0, random_state=0)
)

# 5) Fit on TRAIN only
ridge_model.fit(X_train, y_train)

# 6) Predict on TRAIN and TEST
y_train_pred = ridge_model.predict(X_train)
y_test_pred = ridge_model.predict(X_test)

# 7) Helper to print metrics
def print_metrics(split_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    print(f"{split_name:>5} | MAE: {mae:6.3f} | RMSE: {rmse:6.3f} | R²: {r2:6.3f}")

print_metrics("Train", y_train, y_train_pred)
print_metrics("Test ", y_test, y_test_pred)


# 7.1 Build model configurations for the six lowest‑MAE models

In [ ]:
# 1) Identify the six models with lowest MAE from Stage 1
top_k = 6
top6 = results_df.sort_values("mae")["model"].head(top_k).tolist()

# 2) Define a complexity ranking (lower = simpler)
complexity_rank = {
    "Ridge": 1,
    "LinearRegression": 2,
    "ExtraTrees": 3,
    "RandomForest": 4,
    "GradientBoosting": 5,
    "XGBRegressor": 6,
    "LGBMRegressor": 7,
    "CatBoostRegressor": 8,
    "MLP": 9,
    "SVR_rbf": 10,
    "KNN": 11,
    "Lasso": 12,
}

# 3) Templates: mapping Stage-1 model name → (model, param_grid, use_pca)
model_templates = {
    "Ridge": {
        "model": Ridge(),
        "param_grid": {
            "regressor__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
        },
        "use_pca": False,
    },
    "LinearRegression": {
        # Very little to tune; this keeps it in the game as a baseline
        "model": LinearRegression(),
        "param_grid": {
            "regressor__fit_intercept": [True, False],
        },
        "use_pca": False,
    },
    "ExtraTrees": {
        "model": ExtraTreesRegressor(
            n_jobs=num_cores,
            random_state=42,
            criterion="absolute_error",
        ),
        "param_grid": {
            "regressor__n_estimators": [300, 500],
            "regressor__max_depth": [15, 20, None],
            "regressor__min_samples_split": [2, 5],
            "regressor__min_samples_leaf": [1, 2],
        },
        "use_pca": False,
    },
    "RandomForest": {
        "model": RandomForestRegressor(
            n_jobs=num_cores,
            random_state=42,
            criterion="absolute_error",
        ),
        "param_grid": {
            "regressor__n_estimators": [300, 500, 700],
            "regressor__max_depth": [15, 20, 25, None],
            "regressor__min_samples_split": [2, 5, 10],
            "regressor__min_samples_leaf": [1, 2, 4],
            "regressor__max_features": ["sqrt", "log2", 0.8],
        },
        "use_pca": False,
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(
            loss="absolute_error",
            random_state=42,
        ),
        "param_grid": {
            "regressor__n_estimators": [300, 500, 700],
            "regressor__max_depth": [4, 6, 8],
            "regressor__learning_rate": [0.01, 0.05, 0.1],
            "regressor__subsample": [0.8, 0.9],
            "regressor__min_samples_split": [2, 5],
            "regressor__min_samples_leaf": [1, 2],
        },
        "use_pca": False,
    },
    "XGBRegressor": {
        "model": xgb.XGBRegressor(
            objective="reg:absoluteerror",
            n_jobs=num_cores,
            random_state=42,
            tree_method="hist",
        ),
        "param_grid": {
            "regressor__n_estimators": [500, 800, 1000],
            "regressor__max_depth": [5, 7, 9],
            "regressor__learning_rate": [0.01, 0.05, 0.1],
            "regressor__subsample": [0.8, 0.9],
            "regressor__colsample_bytree": [0.8, 0.9],
            "regressor__min_child_weight": [1, 3, 5],
            "regressor__gamma": [0, 0.1, 0.2],
        },
        "use_pca": False,
    },
    "LGBMRegressor": {
        "model": lgb.LGBMRegressor(
            objective="mae",
            n_jobs=num_cores,
            random_state=42,
            verbose=-1,
        ),
        "param_grid": {
            "regressor__n_estimators": [500, 800, 1000],
            "regressor__max_depth": [5, 7, 9, -1],
            "regressor__learning_rate": [0.01, 0.05, 0.1],
            "regressor__num_leaves": [31, 50, 70],
            "regressor__min_child_samples": [20, 30, 50],
            "regressor__subsample": [0.8, 0.9],
            "regressor__colsample_bytree": [0.8, 0.9],
        },
        "use_pca": False,
    },
    "MLP": {
        "model": MLPRegressor(
            random_state=42,
            max_iter=400,
        ),
        "param_grid": {
            "regressor__hidden_layer_sizes": [(50,), (100,), (100, 50)],
            "regressor__alpha": [1e-5, 1e-4, 1e-3],
            "regressor__learning_rate_init": [0.001, 0.01],
        },
        "use_pca": False,
    },
}

# 4) Build model_configs: ONLY the top-6, ordered from less → more complex
model_configs = {}
for name in sorted(top6, key=lambda m: complexity_rank.get(m, 999)):
    if name in model_templates:
        model_configs[name] = model_templates[name]

# model_configs is now an ordered dict-like (in insertion order):
# e.g. {"Ridge": {...}, "LinearRegression": {...}, "ExtraTrees": {...}, "XGBRegressor": {...}, ...}


# 7.2 Tune the top‑6 models on each dataset and collect validation metrics

In [ ]:
tuned_results = []
best_models = {}

# Target column for the current horizon (e.g. "24h")
target_col = horizons[current_horizon]
target_cols_all = list(horizons.values())

for ds_name, df in datasets.items():
    print("\n" + "="*80)
    print(f"DATASET: {ds_name} – tuning top-6 models for target {target_col}")
    print("="*80 + "\n")

    # -----------------------------
    # 1) Build X, y for this dataset
    # -----------------------------
    df_model = df.copy()

    # Drop ALL target columns from X (we only predict one at a time)
    X = df_model.drop(columns=target_cols_all)
    y = df_model[target_col]

    # Train / validation split
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # -----------------------------
    # 3) Tune each of the top-6 models
    # -----------------------------
    for model_name, cfg in model_configs.items():
        print(f"\n--- Tuning {model_name} on dataset '{ds_name}' ---")

        base_regressor = cfg["model"]
        use_pca = cfg.get("use_pca", False)

        if use_pca:
            model = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("pca", PCA()),
                ("regressor", base_regressor),
            ])
            param_distributions = cfg["param_grid"].copy()
            # If you ever add "pca__n_components" to the grid, it will work here
        else:
            model = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("regressor", base_regressor),
            ])
            param_distributions = cfg["param_grid"]

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_distributions,
            n_iter=25,
            scoring="neg_mean_absolute_error",
            cv=3,
            random_state=42,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )

        search.fit(X_train, y_train)
        best_estimator = search.best_estimator_

        # Store the tuned pipeline, keyed by (dataset, model_name)
        best_models[(ds_name, model_name)] = best_estimator

        # Evaluate tuned model on validation set
        y_pred_valid = best_estimator.predict(X_valid)
        mae_valid = mean_absolute_error(y_valid, y_pred_valid)
        mse_valid = mean_squared_error(y_valid, y_pred_valid)
        rmse_valid = mse_valid ** 0.5
        r2_valid = r2_score(y_valid, y_pred_valid)

        tuned_results.append({
            "dataset": ds_name,
            "model": model_name,
            "cv_best_mae": -search.best_score_,
            "val_mae": mae_valid,
            "val_rmse": rmse_valid,
            "val_r2": r2_valid,
            "best_params": search.best_params_,
        })

# ------------------------------------------------------------
# Collect and inspect results across ALL datasets
# ------------------------------------------------------------
tuned_results_df = (
    pd.DataFrame(tuned_results)
    .sort_values(by=["val_mae", "val_rmse"])
    .reset_index(drop=True)
)

print("\n" + "="*80)
print("TUNED MODELS – VALIDATION PERFORMANCE ACROSS ALL DATASETS")
print("="*80 + "\n")

display(tuned_results_df[["dataset", "model", "cv_best_mae", "val_mae", "val_rmse", "val_r2"]])

print("\nBest combinations (overall, by validation MAE):")
display(tuned_results_df.head(10)[["dataset", "model", "val_mae", "val_rmse", "val_r2"]])


## 7.3 Creating a summary table

In [ ]:
# If for some reason tuned_results_df does not exist yet, build it
if "tuned_results_df" not in locals():
    tuned_results_df = pd.DataFrame(tuned_results)

# Select and rename columns for clarity
summary_table = (
    tuned_results_df[["model", "cv_best_mae", "val_mae", "val_rmse", "val_r2"]]
    .rename(columns={
        "model": "Model",
        "cv_best_mae": "CV_MAE_Mean",
        "val_mae": "Val_MAE",
        "val_rmse": "Val_RMSE",
        "val_r2": "Val_R2"
    })
)

# Sort from lowest to highest validation MAE (and RMSE as tie-breaker)
summary_table = (
    summary_table
    .sort_values(by=["Val_MAE", "Val_RMSE"])
    .reset_index(drop=True)
)

# Round numeric columns
summary_table[["CV_MAE_Mean", "Val_MAE", "Val_RMSE", "Val_R2"]] = \
    summary_table[["CV_MAE_Mean", "Val_MAE", "Val_RMSE", "Val_R2"]].round(3)

print("\n" + "="*80)
print("TUNED MODELS – VALIDATION PERFORMANCE (sorted by Val_MAE)")
print("="*80 + "\n")

display(summary_table)


# 8. Create Ensemble from top perfoming models 

In [ ]:
print("\n" + "=" * 80)
print("CREATING ENSEMBLE MODEL")
print("=" * 80 + "\n")

# ------------------------------------------------------------------
# 8.1  Identify the best single model (by validation MAE)
# ------------------------------------------------------------------
best_row = tuned_results_df.iloc[0]          # row with lowest Val_MAE
best_name = best_row["model"]
best_estimator = best_models[best_name]

best_model_info = {
    "pipeline": best_estimator,
    "model_name": best_name,
    "dataset": "train_split",               # can update later if you retrain on full data
    "metrics": {"val_mae": best_row["val_mae"]},
}
best_val_mae = best_row["val_mae"]

# ------------------------------------------------------------------
# 8.2  Use the existing train split for ensemble training
# ------------------------------------------------------------------
# We keep X_train / y_train as the training set for the ensemble
X = X_train.copy()
y = y_train.copy()

# If you prefer, you *could* retrain on the full df_model later (train + valid)

# ------------------------------------------------------------------
# 8.3  Define base estimators from tuned models
# ------------------------------------------------------------------
base_estimators = []

def _regressor_from_pipeline(pipeline):
    """Extract the final regressor step from a sklearn Pipeline."""
    return pipeline.named_steps["regressor"]

# Take the top-N tuned models as base learners (here N=3)
top_n = 3
for _, row in tuned_results_df.head(top_n).iterrows():
    name = row["model"]
    est = best_models[name]
    base_estimators.append((name, _regressor_from_pipeline(est)))

# (Optional) add *extra* base learners with fixed hyperparameters
# but only if you really want a larger ensemble.
# Make sure the flags match what you defined earlier (xgb_available / lgb_available).

if "xgb_available" in globals() and xgb_available:
    base_estimators.append(
        (
            "xgb_extra",
            xgb.XGBRegressor(
                n_estimators=800,
                max_depth=7,
                learning_rate=0.05,
                objective="reg:absoluteerror",
                n_jobs=num_cores,
                random_state=42,
            ),
        )
    )

if "lgb_available" in globals() and lgb_available:
    base_estimators.append(
        (
            "lgb_extra",
            lgb.LGBMRegressor(
                n_estimators=800,
                max_depth=7,
                learning_rate=0.05,
                objective="mae",
                n_jobs=num_cores,
                random_state=42,
                verbose=-1,
            ),
        )
    )

# ------------------------------------------------------------------
# 8.4  Meta-learner and stacking regressor
# ------------------------------------------------------------------
meta_learner = Ridge(alpha=1.0)

stacking_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=num_cores,
    passthrough=False,
)

# IMPORTANT: reuse the same preprocessor logic as before
ensemble_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("regressor", stacking_model),
    ]
)

# ------------------------------------------------------------------
# 8.5  Train the ensemble
# ------------------------------------------------------------------
print(f"Training ensemble with {len(base_estimators)} base models...")
start_time = time.time()
ensemble_pipeline.fit(X, y)
training_time = time.time() - start_time

# ------------------------------------------------------------------
# 8.6  Evaluate on the original validation split
# ------------------------------------------------------------------
y_val_pred = ensemble_pipeline.predict(X_valid)
ensemble_val_mae = mean_absolute_error(y_valid, y_val_pred)
ensemble_val_rmse = np.sqrt(mean_squared_error(y_valid, y_val_pred))
ensemble_val_r2 = r2_score(y_valid, y_val_pred)

print("\n✅ Ensemble training complete!")
print(f"   MAE (validation): {ensemble_val_mae:.3f}")
print(f"   RMSE (validation): {ensemble_val_rmse:.3f}")
print(f"   R² (validation): {ensemble_val_r2:.3f}")
print(f"   Training time: {training_time:.2f} s")

# ------------------------------------------------------------------
# 8.7  Decide whether to keep the ensemble
# ------------------------------------------------------------------
if ensemble_val_mae < best_val_mae:
    improvement = (best_val_mae - ensemble_val_mae) / best_val_mae * 100
    print(f"\n⭐ Ensemble outperforms best single model by {improvement:.1f}%")
    best_model_info["pipeline"] = ensemble_pipeline
    best_model_info["model_name"] = "Ensemble (Stacking)"
    best_model_info["metrics"]["val_mae"] = ensemble_val_mae
else:
    print(
        f"\nSingle model still better. "
        f"Ensemble MAE ({ensemble_val_mae:.3f}) > best MAE ({best_val_mae:.3f})"
    )

print("\n" + "=" * 80)
print("ENSEMBLE MODEL READY – proceed to final training / submission step")
print("=" * 80 + "\n")


# 9. Kaggle Submission creation with EnsembleModel

## 9.1 Loading test data

In [ ]:
# 1) Import the raw test dataset
test = pd.read_csv("ML_WP_data/test.csv")

# 2) Apply the SAME feature engineering as for the training data
#    (this already includes cyclical hour/season features)
test = create_advanced_features(test, is_training=False)

# 3) Quick sanity check: first rows
test.head()

# 4) Count NaNs in the engineered test set
total_test_nans = test.isna().sum().sum()
print("Total NaN values in the DataFrame:", total_test_nans)


## 9.2 Creating the Kaggle submission file

In [ ]:
print("\n" + "="*80)
print("FINAL STEP: REFIT BEST MODEL & GENERATE KAGGLE SUBMISSION")
print("="*80 + "\n")

# 0) Safety check – best_model_info, X_train, X_valid, test must exist
if "best_model_info" not in globals() or best_model_info is None:
    raise ValueError("No best model found. Run the model selection / ensemble cell first.")

if "X_train" not in globals() or "X_valid" not in globals():
    raise ValueError("X_train / X_valid not found. Run the data prep + split cells first.")

if "test" not in globals():
    raise ValueError("Engineered test set 'test' not found. Run the test feature-engineering cell first.")

best_model_name = best_model_info["model_name"]
best_pipeline   = best_model_info["pipeline"]
best_metrics    = best_model_info["metrics"]

# Robustly get a MAE value for logging/filename
holdout_mae = (
    best_metrics.get("test_mae")
    or best_metrics.get("val_mae")
    or best_metrics.get("holdout_mae")
    or best_metrics.get("mae")
)

print("Best overall model (by MAE on hold-out set):", best_model_name)
if holdout_mae is not None:
    print(f"MAE on hold-out set: {holdout_mae:.3f}")
else:
    print("MAE on hold-out set: not available")

# 1) Use the ALREADY ENGINEERED Kaggle test set
print("\nUsing existing engineered Kaggle test set: 'test'")
print("Kaggle test features shape:", test.shape)

total_test_nans = test.isna().sum().sum()
print("Total NaN values in engineered test set:", total_test_nans)
  
# 2) Refit the best model on ALL training data (train + valid)
X_full = pd.concat([X_train, X_valid], axis=0)
y_full = pd.concat([y_train, y_valid], axis=0)

print(f"\nRefitting best model ({best_model_name}) on full training data...")
best_pipeline.fit(X_full, y_full)

# Optional bookkeeping
best_model_info["trained_on"] = "train+valid"

# 3) Predict on Kaggle test (let the pipeline handle columns)
X_kaggle = test.copy()
print("\nGenerating predictions for target_tre200h0_plus24h...")
y_pred = best_pipeline.predict(X_kaggle)

# 4) Build submission DataFrame
if "Id" in test.columns:
    submission_ids = test["Id"]
else:
    submission_ids = range(1, len(test) + 1)

submission = pd.DataFrame({
    "Id": submission_ids,
    "target_tre200h0_plus24h": y_pred,
})

print(f"\nSubmission shape: {submission.shape}")
print("\nPrediction statistics:")
print(submission["target_tre200h0_plus24h"].describe())

# 5) Save submission file
day = date.today().strftime("%Y%m%d")
mae_for_name = holdout_mae if holdout_mae is not None else np.nan
filename = f"weather_submission_improved_{day}_MAE_{mae_for_name:.3f}.csv"

submission.to_csv(filename, index=False, encoding="utf-8")
print(f"\nSubmission saved as: {filename}")
print("\nFirst few predictions:")
display(submission.head(10))

# 6) (Optional) Save the final refitted model to disk
from pathlib import Path
import joblib

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)
model_path = models_dir / f"best_pipeline_{best_model_name}_MAE_{mae_for_name:.3f}.joblib"
joblib.dump(best_pipeline, model_path, compress=3)
print(f"\nFinal model saved to: {model_path}")

print("\n" + "="*80)
print("DONE: MODEL REFITTED, SUBMISSION & MODEL FILE CREATED")
print("="*80 + "\n")
